# WESAD Phase 2: Supervised Fine-Tuning
This notebook contains the complete fine-tuning loop for WESAD, incorporating critical fixes to correctly wrap the model expected parameters, fix dimensions according to channel sampling length, and appropriately interpret binary target labels.

In [1]:
# ==============================================================================
# FASE 2: FINE-TUNING E TESTING SU WESAD
# ==============================================================================
import os, sys, shutil, glob, pickle, numpy as np

# 1. SETUP AMBIENTE
WORK_DIR = '/kaggle/working/e4selflearning'
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
print(" Clonazione repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}
os.chdir(WORK_DIR)

# Installazione librerie
print(" Installazione dipendenze...")
!pip install --no-cache-dir -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biopsykit flirt hrv-analysis neurokit2 peakutils lightning torchmetrics torchinfo scikit-learn

# 2. CREAZIONE SCRIPT DI TRAINING PERSONALIZZATO
print(" Generazione script train_ann_fixed.py...")

script_content = r'''
import argparse, os, pickle, numpy as np, torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from timebase import criterions
from timebase.models.models import get_models
from timebase.data.dataset import ClassificationDataset
from timebase.utils import utils
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

class WESAD_ClassificationDataset(ClassificationDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.positive_classes = [1.0]
        
    def session_id(self, filename: str) -> int:
        basename = os.path.basename(os.path.dirname(filename))
        if basename.startswith("S"):
            return int(basename[1:])
        return int(basename)

def load_pre_trained(args, classifier):
    target_file = None
    for r, d, f in os.walk(args.path2pretraining_res):
        if "model_state.pt" in f:
            target_file = os.path.join(r, "model_state.pt")
            break
    if not target_file:
        raise FileNotFoundError("ERRORE: model_state.pt non trovato nei pre-trained weights!")
    
    ckpt = torch.load(target_file, map_location=args.device)
    sd = classifier.sslearner.state_dict()
    sd.update({k: v for k, v in ckpt["model"].items() if any(m in k for m in ["channel_embedding", "feature_encoder"])})
    classifier.sslearner.load_state_dict(sd)
    print(" [OK] Pesi SSL caricati con successo.")

def main(args):
    utils.set_random_seed(args.seed)
    utils.get_device(args)
    args.task_mode = 1
    args.split_mode = 1
    args.verbose = 1
    args.use_wandb = False
    args.critic_score_lambda = 1.0
    
    os.makedirs(args.output_dir, exist_ok=True)
    
    cli_dataset = getattr(args, "dataset", None)
    args.task_mode = 2 
    utils.load_args(args, dir=args.path2pretraining_res)
    args.task_mode = 1 
    if cli_dataset is not None:
        args.dataset = cli_dataset
    
    with open(os.path.join(args.dataset, "metadata.pkl"), "rb") as f:
        meta = pickle.load(f)
    args.ds_info = meta["ds_info"]
    
    stats = None
    stats_path = os.path.join(args.dataset, "stats.pkl")
    if os.path.exists(stats_path):
        with open(stats_path, "rb") as f:
            stats_loaded = pickle.load(f)
            stats = stats_loaded[list(stats_loaded.keys())[0]]
    if stats is None:
        print(" [!] Stats non trovate, uso valori di default.")
        channels = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
        stats = {c: {'min': -10, 'max': 10, 'mean': 0, 'std': 1, 'median': 0, 'iqr': 1} for c in channels}
    
    args.input_shapes = {c: [512 * args.ds_info["channel_freq"][c]] for c in ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']}
    
    # ------------------ SPLIT TRAIN vs TEST ------------------
    TRAIN_SUBS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11']
    TEST_SUBS  = ['S13', 'S14', 'S15', 'S16', 'S17']
    
    dynamic_paths = []
    for p in meta["sessions_paths"]:
        parts = p.replace("\\\\", "/").split("/")
        dynamic_paths.append(os.path.join(args.dataset, parts[-2], parts[-1]))
    all_paths = np.array(dynamic_paths)
    
    labels_dict = {k: np.array(v) for k, v in meta["sessions_labels"].items()}
    mask_train = np.array([any(s in p for s in TRAIN_SUBS) for p in all_paths])
    mask_test  = np.array([any(s in p for s in TEST_SUBS) for p in all_paths])
    
    # Generazione Subset Train
    f_train = all_paths[mask_train]
    l_train = {k: v[mask_train] for k, v in labels_dict.items()}
    unique_recs = np.unique(f_train)
    rec_to_num = {path: i for i, path in enumerate(unique_recs)}
    args.num_train_subjects = len(TRAIN_SUBS)
    
    print(f" Dataset Addestramento pronto: {len(f_train)} segmenti.")
    ds = WESAD_ClassificationDataset(
        args=args, 
        filenames=f_train, 
        labels=l_train,
        rec_ids=f_train,
        stats=stats,
        recording_id_str_to_num=rec_to_num,
        subject_ids=l_train.get("Sub_ID", None)
    )
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True)
    
    classifier, critic = get_models(args, summary=None)
    load_pre_trained(args, classifier)
    
    classifier.to(args.device).train()
    classifier.sslearner.requires_grad_(False)
    if critic is not None:
        critic.to(args.device).train()
    
    opt = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    crit = criterions.get_criterion(args)
    
    # ================== TRAINING LOOP ==================
    for epoch in range(1, args.epochs + 1):
        loss_epoch = []
        for batch in tqdm(loader, desc=f"Epoca {epoch}", leave=False):
            inputs = {k: v.to(args.device) for k, v in batch["data"].items()}
            targets = batch["target"].to(args.device)
            out, _ = classifier(inputs)
            loss = crit.criterion_classifier(y_true=targets, y_pred=out)
            loss.backward(); opt.step(); opt.zero_grad()
            loss_epoch.append(loss.item())
        print(f" Epoca {epoch}: Loss = {np.mean(loss_epoch):.4f}")

    # Salva il modello a fine addestramento
    out_model_path = os.path.join(args.output_dir, "classifier_finetuned.pt")
    torch.save(classifier.state_dict(), out_model_path)
    print(f"\n [SALVATAGGIO] Modello salvato in: {out_model_path}")
    
    # ================== TESTING LOOP ==================
    print("\n--- INIZIO FASE DI VALUTAZIONE ---")
    f_test = all_paths[mask_test]
    l_test = {k: v[mask_test] for k, v in labels_dict.items()}
    
    # Mappa temporanea finta per il test (evita crash se un record test ha path nuovo)
    rec_to_num_test = {path: i for i, path in enumerate(np.unique(f_test))}
    
    ds_test = WESAD_ClassificationDataset(
        args=args, 
        filenames=f_test, 
        labels=l_test,
        rec_ids=f_test,
        stats=stats,
        recording_id_str_to_num=rec_to_num_test,
        subject_ids=l_test.get("Sub_ID", None)
    )
    test_loader = DataLoader(ds_test, batch_size=args.batch_size, shuffle=False)
    
    classifier.eval()
    all_targets = []
    all_preds_probs = []
    all_preds_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Esecuzione Valutazione"):
            inputs = {k: v.to(args.device) for k, v in batch["data"].items()}
            targets = batch["target"].to(args.device)
            out, _ = classifier(inputs)
            # Trasformazione logit in probabilità binaria
            probs = torch.sigmoid(out).squeeze()
            preds = (probs >= 0.5).long()
            
            # Appiattimento dimensioni
            if targets.dim() > 1:
                targets = targets.squeeze()
            
            # Gestione casistica ultimo batch da 1 singolo segmento
            if probs.dim() == 0:
                probs = probs.unsqueeze(0)
                preds = preds.unsqueeze(0)
                targets = targets.unsqueeze(0)
                
            all_targets.extend(targets.cpu().numpy())
            all_preds_probs.extend(probs.cpu().numpy())
            all_preds_labels.extend(preds.cpu().numpy())
            
    # Calcolo Metriche
    acc = accuracy_score(all_targets, all_preds_labels)
    f1 = f1_score(all_targets, all_preds_labels, zero_division=0)
    prec = precision_score(all_targets, all_preds_labels, zero_division=0)
    rec = recall_score(all_targets, all_preds_labels, zero_division=0)
    cm = confusion_matrix(all_targets, all_preds_labels)
    
    print("\n\n" + "="*40)
    print(" RAPPORTO E METRICHE DI CLASSIFICAZIONE")
    print("="*40)
    print(f" Datasets Pazienti: S13, S14, S15, S16, S17")
    print(f" Numero Totale Segmenti Analizzati: {len(all_targets)}")
    print(f"----------------------------------------")
    print(f" ACCURACY  Globale  : {acc*100:>6.2f}%")
    print(f" F1-SCORE  (Stress) : {f1*100:>6.2f}%")
    print(f" PRECISION (Stress) : {prec*100:>6.2f}%")
    print(f" RECALL    (Stress) : {rec*100:>6.2f}%")
    print(f"----------------------------------------")
    print(" MATRICE DI CONFUSIONE:")
    if cm.shape == (2, 2):
        print(f"  Vero Neutrale (TN) : {cm[0][0]:<5} | Falso Stress   (FP) : {cm[0][1]:<5}")
        print(f"  Falsi Negativi (FN): {cm[1][0]:<5} | Vero Stress    (TP) : {cm[1][1]:<5}")
    else:
        print(f"  Dati Monoclasse, CM fallback:\n {cm}")
    print("="*40)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, required=True)
    parser.add_argument("--path2pretraining_res", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=50) # Tieni a 50
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=0.0001)
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--seed", type=int, default=1234)
    parser.add_argument("--output_dir", type=str, default="output")
    main(parser.parse_args())
'''

with open("train_ann_fixed.py", "w") as f:
    f.write(script_content)

wesad_meta = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
DATA_DIR = os.path.dirname(wesad_meta[0])
ckpt_dirs = glob.glob('/kaggle/input/**/ckpt_sslearner', recursive=True)
if not ckpt_dirs:
    print("ERRORE CRITICO: Non sono stati trovati i pesi di pretrained model 'ckpt_sslearner'")
else:
    PRE_RES = os.path.dirname(ckpt_dirs[0])
    print(f"Pesi pre-trained trovati in {PRE_RES}")

!PYTHONPATH=src python train_ann_fixed.py \
    --dataset {DATA_DIR} \
    --path2pretraining_res {PRE_RES} \
    --epochs 50 \
    --output_dir output_wesad_final


 Clonazione repository...
Cloning into '/kaggle/working/e4selflearning'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 71 (delta 12), reused 59 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 94.89 KiB | 1.86 MiB/s, done.
Resolving deltas: 100% (12/12), done.
 Installazione dipendenze...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 291.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 216.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 312.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 306.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 287.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━